# Notebook OSeMOSYS (standalone) v2 — flujo completo con validación de calidad

Notebook **autocontenido** que ejecuta el flujo OSeMOSYS sin importar funciones de `app.simulation.core.*` (excepto `create_abstract_model` por su tamaño). Incluye **el sistema de validación de calidad de datos inline**.

Funciona desde:
- **Excel SAND** (hoja `Parameters`)
- **Carpeta de CSVs** ya procesados

El flujo:
1. Configuración (paths)
2. Helpers SAND → CSV (inline)
3. Pipeline run
4. **Validación de calidad** — bound_conflicts + year_exclusions inline, con auto-fix
5. AbstractModel (inline completo)
6. build_instance (inline)
7. Solve (Gurobi → HiGHS → GLPK) con dispose de Env
8. **Diagnóstico de infactibilidad**
9. **Resultados como DataFrames** estilo data explorer (filtrable)

## 1. Configuración (paths)

In [ ]:
import os
from pathlib import Path

EXCEL_PATH: Path | None = Path("/Users/davidbedoya0/Downloads/29_04_26_SAND_INTEGRADO_CN_V2 _RM__Parameters_SAND.xlsx")
CSV_DIR_IN: Path | None = None
CSV_DIR_OUT: Path = Path.cwd() / "tmp_osemosys_v2"

GUROBI_LIC_FILE: Path | None = Path("/Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/secrets/gurobi.lic")
if GUROBI_LIC_FILE is not None and GUROBI_LIC_FILE.is_file():
    os.environ["GRB_LICENSE_FILE"] = str(GUROBI_LIC_FILE)

assert (EXCEL_PATH is None) ^ (CSV_DIR_IN is None), "Activa solo una fuente"
print("EXCEL_PATH :", EXCEL_PATH)
print("CSV_DIR_IN :", CSV_DIR_IN)
print("CSV_DIR_OUT:", CSV_DIR_OUT)
print("GRB_LICENSE:", os.environ.get("GRB_LICENSE_FILE", "(no set)"))

## 2. Helpers SAND → CSV (inline)

In [ ]:
import os, itertools
from itertools import product
import numpy as np
import pandas as pd


def SAND_to_CSV(df, param, path_csv, div):
    df_param = df[df["Parameter"] == param].dropna(axis=1)
    year = df_param.columns[df_param.columns.to_series().apply(pd.to_numeric, errors="coerce").notna()]
    sets = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()
    if "Time indipendent variables" in sets:
        sets.remove("Time indipendent variables")
        df_TUPLES = df_param.drop(["Parameter"], axis=1).rename(columns={"Time indipendent variables": "VALUE"})
    elif "TIMESLICE" in sets:
        df_param = df_param.reset_index(drop=True)
        df_param_sub = df_param[df_param.index % div == 0].reset_index(drop=True)
        df_param_sub = df_param_sub.set_index(sets).drop(columns="Parameter")
        if param == "CapacityFactor":
            df_param_sub = df_param_sub.reset_index()
            df_agg = df_param.drop(columns=["Parameter"] + sets)
            df_agg["index_col"] = df_agg.index // div
            df_agg = df_agg.groupby("index_col", as_index=False).mean()
            for y in year: df_param_sub[y] = df_agg[y]
            df_param_indexed = df_param_sub.set_index(sets)
        else:
            df_param_sub = df_param_sub.loc[~(df_param_sub == 0).all(axis=1)].reset_index()
            df_agg = df_param.drop(columns=["Parameter"] + sets).loc[~(df_param.drop(columns=["Parameter"] + sets) == 0).all(axis=1)]
            df_agg["index_col"] = df_agg.index // div
            df_agg = df_agg.groupby("index_col", as_index=False).sum()
            for y in year: df_param_sub[y] = df_agg[y]
            df_param_indexed = df_param_sub.set_index(sets)
        index_product = list(product(df_param_indexed.index, year))
        df_TUPLES = pd.DataFrame(index=index_product, columns=["VALUE"])
        for idx in df_TUPLES.index:
            df_TUPLES.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
        df_TUPLES.reset_index(inplace=True)
        df_TUPLES[["SETS", "YEAR"]] = pd.DataFrame(df_TUPLES["index"].tolist(), index=df_TUPLES.index)
        df_TUPLES[sets] = pd.DataFrame(df_TUPLES["SETS"].tolist(), index=df_TUPLES.index)
        df_TUPLES = df_TUPLES.drop(["index", "SETS"], axis=1)
        df_TUPLES = df_TUPLES[sets + ["YEAR", "VALUE"]]
    else:
        df_param_indexed = df_param.set_index(sets).drop(columns="Parameter")
        if df_param_indexed.empty:
            df_TUPLES = pd.DataFrame(columns=sets + ["YEAR", "VALUE"])
        else:
            index_product = list(product(df_param_indexed.index, year))
            df_TUPLES = pd.DataFrame(index=index_product, columns=["VALUE"])
            for idx in df_TUPLES.index:
                df_TUPLES.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
            df_TUPLES.reset_index(inplace=True)
            df_TUPLES[["SETS", "YEAR"]] = pd.DataFrame(df_TUPLES["index"].tolist(), index=df_TUPLES.index)
            df_TUPLES[sets] = pd.DataFrame(df_TUPLES["SETS"].tolist(), index=df_TUPLES.index)
            df_TUPLES = df_TUPLES.drop(["index", "SETS"], axis=1)
            df_TUPLES = df_TUPLES[sets + ["YEAR", "VALUE"]]
            df_TUPLES = df_TUPLES.dropna(axis=1)
    df_TUPLES.to_csv(os.path.join(path_csv, f"{param}.csv"), index=False)


def SAND_SETS_to_CSV(df, path_csv, div):
    df_param = df[df["Parameter"] == "YearSplit"].dropna(axis=1).reset_index(drop=True)
    df_param = df_param[df_param.index % div == 0]
    year = df_param.columns[df_param.columns.to_series().apply(pd.to_numeric, errors="coerce").notna()]
    df_year = pd.DataFrame(year, columns=["VALUE"])
    sets = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()
    for s in sets:
        pd.DataFrame(df_param[s].unique(), columns=["VALUE"]).to_csv(os.path.join(path_csv, f"{s}.csv"), index=False)
    df_year.to_csv(os.path.join(path_csv, "YEAR.csv"), index=False)
    for src_param, target_dims, also_set in [
        ("EmissionActivityRatio", ["EMISSION"], True),
        ("OutputActivityRatio", None, True),
        ("CapacityToActivityUnit", ["REGION", "TECHNOLOGY"], False),
    ]:
        df_param = df[df["Parameter"] == src_param].dropna(axis=1)
        if df_param.empty: continue
        sets_src = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()
        if "Time indipendent variables" in sets_src:
            sets_src.remove("Time indipendent variables")
            df_t = df_param.drop(["Parameter"], axis=1).rename(columns={"Time indipendent variables": "VALUE"}).loc[
                lambda x: x["VALUE"] != 0
            ]
        else:
            df_param_indexed = df_param.set_index(sets_src).drop(columns="Parameter").loc[
                ~(df_param.set_index(sets_src).drop(columns="Parameter") == 0).all(axis=1)
            ]
            if df_param_indexed.empty:
                df_t = pd.DataFrame(columns=sets_src + ["YEAR", "VALUE"])
            else:
                idxp = list(product(df_param_indexed.index, year))
                df_t = pd.DataFrame(index=idxp, columns=["VALUE"])
                for idx in df_t.index: df_t.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
                df_t.reset_index(inplace=True)
                df_t[["SETS", "YEAR"]] = pd.DataFrame(df_t["index"].tolist(), index=df_t.index)
                df_t[sets_src] = pd.DataFrame(df_t["SETS"].tolist(), index=df_t.index)
                df_t = df_t.drop(["index", "SETS"], axis=1)
                df_t = df_t[sets_src + ["YEAR", "VALUE"]]
        targets = target_dims if target_dims else sets_src
        for s in targets:
            if s in df_t.columns:
                pd.DataFrame(df_t[s].unique(), columns=["VALUE"]).to_csv(os.path.join(path_csv, f"{s}.csv"), index=False)


def completar_Matrix_Act_Ratio(path_csv, variable):
    df = pd.read_csv(path_csv + variable)
    regions = df["REGION"].unique()
    technologies = pd.read_csv(path_csv + "TECHNOLOGY.csv", dtype=str)["VALUE"].unique()
    fuels = pd.read_csv(path_csv + "FUEL.csv", dtype=str)["VALUE"].unique()
    modes = pd.read_csv(path_csv + "MODE_OF_OPERATION.csv")["VALUE"].unique()
    years = pd.read_csv(path_csv + "YEAR.csv")["VALUE"].unique()
    all_c = pd.DataFrame(itertools.product(regions, technologies, fuels, modes, years),
        columns=["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR"])
    res = all_c.merge(df, on=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "FUEL", "YEAR"], how="left")
    res.dropna(subset=["VALUE"], inplace=True)
    res.to_csv(path_csv + variable, index=False)


def completar_Matrix_Emission(path_csv, variable):
    df = pd.read_csv(path_csv + variable)
    regions = df["REGION"].unique()
    technologies = pd.read_csv(path_csv + "TECHNOLOGY.csv")["VALUE"].unique()
    emission = pd.read_csv(path_csv + "EMISSION.csv")["VALUE"].unique()
    modes = pd.read_csv(path_csv + "MODE_OF_OPERATION.csv")["VALUE"].unique()
    years = pd.read_csv(path_csv + "YEAR.csv")["VALUE"].unique()
    all_c = pd.DataFrame(itertools.product(regions, technologies, emission, modes, years),
        columns=["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"])
    res = all_c.merge(df, on=["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"], how="left")
    res.dropna(subset=["VALUE"], inplace=True)
    res.to_csv(path_csv + variable, index=False)


def completar_Matrix_Cost(path_csv, variable):
    df = pd.read_csv(path_csv + variable)
    regions = df["REGION"].unique()
    technologies = pd.read_csv(path_csv + "TECHNOLOGY.csv")["VALUE"].unique()
    modes = pd.read_csv(path_csv + "MODE_OF_OPERATION.csv")["VALUE"].unique()
    years = pd.read_csv(path_csv + "YEAR.csv")["VALUE"].unique()
    all_c = pd.DataFrame(itertools.product(regions, technologies, modes, years),
        columns=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"])
    res = all_c.merge(df, on=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"], how="left")
    res.dropna(subset=["VALUE"], inplace=True)
    res.to_csv(path_csv + variable, index=False)


def process_and_save_emission_ratios(emission_path, input_path, output_path, path_csv):
    df_e = pd.read_csv(path_csv + emission_path); df_i = pd.read_csv(path_csv + input_path)
    keys = ["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"]
    for d in (df_e, df_i):
        for k in keys:
            if k in d.columns: d[k] = d[k].astype(str).str.strip()
    merged = pd.merge(df_e, df_i, on=keys, how="left").query("VALUE_x != 0 and VALUE_y != 0").assign(
        VALUE=lambda x: x["VALUE_x"] * x["VALUE_y"])
    drop_cols = [c for c in ["VALUE_x", "VALUE_y", "FUEL"] if c in merged.columns]
    merged = merged.drop(columns=drop_cols)
    mu = merged.groupby(["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"], as_index=False).agg({"VALUE": "first"})
    fm = pd.merge(df_e, mu, on=["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"], how="left", suffixes=("_df1", "_df4")).assign(
        VALUE=lambda x: x.apply(lambda r: r["VALUE_df4"] if pd.notnull(r["VALUE_df4"]) and r["VALUE_df1"] != r["VALUE_df4"] else r["VALUE_df1"], axis=1)
    ).loc[:, ["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR", "VALUE"]]
    fm.to_csv(path_csv + output_path, index=False)


def filter_params_by_sets(path_csv, df_colombia):
    for p in df_colombia["Parameter"].unique():
        fp = path_csv + p + ".csv"
        if not os.path.exists(fp): continue
        df_p = pd.read_csv(fp)
        sc = df_p.columns.drop("VALUE")
        if "REGION2" in sc: sc = df_p.columns.drop(["REGION2", "VALUE"])
        for s in sc:
            sf = path_csv + s + ".csv"
            if os.path.exists(sf):
                df_p = df_p[df_p[s].isin(pd.read_csv(sf).VALUE.tolist())]
        df_p.to_csv(fp, index=False)

print("Helpers SAND-to-CSV definidos.")

## 3. Generar CSVs

In [ ]:
import time, shutil

if EXCEL_PATH is not None:
    if CSV_DIR_OUT.exists(): shutil.rmtree(CSV_DIR_OUT)
    CSV_DIR_OUT.mkdir(parents=True, exist_ok=True)
    path_csv = str(CSV_DIR_OUT) + os.sep
    div = 1
    t0 = time.perf_counter()
    df = pd.read_excel(EXCEL_PATH, sheet_name="Parameters")
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    print(f"Excel leido: {df['Parameter'].nunique()} parametros")

    SAND_SETS_to_CSV(df, path_csv, 96 / div)
    for p in df["Parameter"].dropna().unique():
        SAND_to_CSV(df, p, path_csv, 96 / div)
    filter_params_by_sets(path_csv, df)
    completar_Matrix_Act_Ratio(path_csv, "InputActivityRatio.csv")
    completar_Matrix_Act_Ratio(path_csv, "OutputActivityRatio.csv")
    if os.path.exists(path_csv + "EMISSION.csv"):
        completar_Matrix_Emission(path_csv, "EmissionActivityRatio.csv")
    completar_Matrix_Cost(path_csv, "VariableCost.csv")
    if os.path.exists(path_csv + "EMISSION.csv"):
        process_and_save_emission_ratios("EmissionActivityRatio.csv", "InputActivityRatio.csv",
            "EmissionActivityRatio.csv", path_csv)
    csv_dir = CSV_DIR_OUT
    _n_csvs = len(list(csv_dir.glob("*.csv")))
    print(f"\nGeneracion completada en {time.perf_counter()-t0:.2f} s -> {_n_csvs} CSVs")
else:
    csv_dir = CSV_DIR_IN
    print(f"Usando CSVs existentes: {csv_dir}")

## 4. Validación de calidad de datos (inline)

Detecta `bound_conflicts` (clasificados real_conflict / numeric_precision con gap < 1e-4) y `year_exclusions` (años con YearSplit=0 en TODOS sus timeslices). El **dead_year exclusion** se aplica automáticamente; el **auto-fix de numeric_precision** es opt-in.

In [ ]:
"""data_validation inline (mirror del modulo del repo)."""
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

NUMERIC_PRECISION_TOL: float = 1e-4
BOUND_PAIRS = (
    ("TotalTechnologyAnnualActivityLowerLimit", "TotalTechnologyAnnualActivityUpperLimit"),
    ("TotalAnnualMinCapacity", "TotalAnnualMaxCapacity"),
    ("TotalAnnualMinCapacityInvestment", "TotalAnnualMaxCapacityInvestment"),
)


@dataclass
class BoundConflict:
    lower: str; upper: str; key: dict
    value_lower: float; value_upper: float; gap: float; severity: str


@dataclass
class YearExclusion:
    year: int; reason: str
    n_timeslices_zero: int; n_timeslices_total: int


def detect_bound_conflicts(csv_dir, *, tol=NUMERIC_PRECISION_TOL):
    csv_dir = Path(csv_dir)
    out = []
    for lower_name, upper_name in BOUND_PAIRS:
        lp, up = csv_dir / f"{lower_name}.csv", csv_dir / f"{upper_name}.csv"
        if not (lp.exists() and up.exists()): continue
        df_l, df_u = pd.read_csv(lp), pd.read_csv(up)
        if df_l.empty or df_u.empty or "VALUE" not in df_l.columns: continue
        kc = [c for c in df_l.columns if c != "VALUE"]
        for c in kc:
            if c == "YEAR":
                df_l[c] = pd.to_numeric(df_l[c], errors="coerce").astype("Int64")
                df_u[c] = pd.to_numeric(df_u[c], errors="coerce").astype("Int64")
            else:
                df_l[c] = df_l[c].astype(str); df_u[c] = df_u[c].astype(str)
        merged = df_l.merge(df_u, on=kc, suffixes=("_LOW", "_UPP"))
        violating = merged[merged["VALUE_LOW"] > merged["VALUE_UPP"]]
        for _, r in violating.iterrows():
            gap = float(r["VALUE_LOW"]) - float(r["VALUE_UPP"])
            key = {c: (r[c].item() if hasattr(r[c], "item") else r[c]) for c in kc}
            severity = "numeric_precision" if abs(gap) < tol else "real_conflict"
            out.append(BoundConflict(lower_name, upper_name, key,
                float(r["VALUE_LOW"]), float(r["VALUE_UPP"]), gap, severity))
    return out


def detect_dead_years(csv_dir):
    csv_dir = Path(csv_dir)
    fp = csv_dir / "YearSplit.csv"
    if not fp.exists(): return []
    df = pd.read_csv(fp)
    if df.empty or "YEAR" not in df.columns or "VALUE" not in df.columns: return []
    df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce").fillna(0.0)
    df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce").astype("Int64")
    grp = df.groupby("YEAR").agg(n_total=("VALUE", "size"), n_zero=("VALUE", lambda v: int((v==0).sum()))).reset_index()
    return [YearExclusion(int(r["YEAR"]), "YearSplit_all_zero", int(r["n_zero"]), int(r["n_total"]))
            for _, r in grp.iterrows() if int(r["n_total"]) > 0 and int(r["n_zero"]) == int(r["n_total"])]


def apply_dead_year_exclusion(csv_dir, exclusions):
    if not exclusions: return 0
    csv_dir = Path(csv_dir); dead = {e.year for e in exclusions}; n = 0
    yr = csv_dir / "YEAR.csv"
    if yr.exists():
        df = pd.read_csv(yr)
        if "VALUE" in df.columns:
            df["VALUE"] = pd.to_numeric(df["VALUE"], errors="coerce").astype("Int64")
            before = len(df); df = df[~df["VALUE"].isin(dead)]
            if len(df) < before: df.to_csv(yr, index=False); n += 1
    for cp in csv_dir.glob("*.csv"):
        if cp.name == "YEAR.csv": continue
        try: head = pd.read_csv(cp, nrows=0)
        except Exception: continue
        if "YEAR" not in head.columns: continue
        df = pd.read_csv(cp)
        if df.empty: continue
        df["YEAR"] = pd.to_numeric(df["YEAR"], errors="coerce").astype("Int64")
        before = len(df); df = df[~df["YEAR"].isin(dead)]
        if len(df) < before: df.to_csv(cp, index=False); n += 1
    return n


def apply_bound_fix_numeric_precision(csv_dir, conflicts):
    if not conflicts: return 0
    csv_dir = Path(csv_dir)
    fix = [c for c in conflicts if c.severity == "numeric_precision"]
    if not fix: return 0
    by_lower = {}
    for c in fix: by_lower.setdefault(c.lower, []).append(c)
    n = 0
    for lower_name, group in by_lower.items():
        lp = csv_dir / f"{lower_name}.csv"
        if not lp.exists(): continue
        df = pd.read_csv(lp)
        if df.empty or "VALUE" not in df.columns: continue
        for col in df.columns:
            if col == "YEAR": df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
            elif col != "VALUE": df[col] = df[col].astype(str)
        for c in group:
            mask = pd.Series([True]*len(df))
            for k, v in c.key.items():
                if k not in df.columns: mask = pd.Series([False]*len(df)); break
                if k == "YEAR": mask &= df[k] == int(v)
                else: mask &= df[k] == str(v)
            if mask.any():
                df.loc[mask, "VALUE"] = c.value_upper
                n += int(mask.sum())
        df.to_csv(lp, index=False)
    return n


print("Validacion inline definida.")

In [ ]:
# Detectar y aplicar dead_years (exclusion automatica) y reportar bound_conflicts.
bound_conflicts = detect_bound_conflicts(csv_dir)
year_exclusions = detect_dead_years(csv_dir)

print("=" * 70)
print("REPORTE DE CALIDAD DE DATOS")
print("=" * 70)

n_real = sum(1 for c in bound_conflicts if c.severity == "real_conflict")
n_prec = sum(1 for c in bound_conflicts if c.severity == "numeric_precision")
print(f"  bound_conflicts: {len(bound_conflicts)} ({n_real} real, {n_prec} numeric_precision)")
print(f"  year_exclusions: {len(year_exclusions)}")
print()

# Aplicar dead_year exclusion (siempre)
n_files = apply_dead_year_exclusion(csv_dir, year_exclusions)
if year_exclusions:
    print(f"Anos excluidos: {[e.year for e in year_exclusions]}")
    print(f"Archivos modificados: {n_files}")
    print()

# Tablas detalladas
if bound_conflicts:
    print("BOUND CONFLICTS:")
    rows = []
    for c in bound_conflicts:
        row = {"lower": c.lower, "upper": c.upper, "value_lower": c.value_lower,
               "value_upper": c.value_upper, "gap": c.gap, "severity": c.severity}
        for k, v in c.key.items(): row[k] = v
        rows.append(row)
    df_bc = pd.DataFrame(rows)
    with pd.option_context("display.width", 200, "display.max_colwidth", 50):
        print(df_bc.to_string(index=False))
    print()

# Auto-fix opt-in (solo numeric_precision)
APPLY_AUTOFIX = True
if APPLY_AUTOFIX and n_prec > 0:
    n_fixed = apply_bound_fix_numeric_precision(csv_dir, bound_conflicts)
    print(f"\nAuto-fix aplicado: {n_fixed} tuplas numeric_precision corregidas (lower = upper).")

# Resumen para downstream
has_real_conflicts = n_real > 0
print(f"\nhas_real_conflicts = {has_real_conflicts}")
if has_real_conflicts:
    print("ATENCION: hay conflictos reales que requieren tu intervencion antes de simular.")

## 5. AbstractModel (inline completo)

In [ ]:
"""Definición completa del AbstractModel OSeMOSYS.

Replica fielmente la celda 3 del notebook osemosys_notebook_UPME_OPT_01.ipynb:
sets, parámetros, variables, función objetivo y TODAS las restricciones
en un solo archivo.

OSeMOSYS: Open Source energy MOdeling SYStem
Copyright [2010-2015] [OSeMOSYS Forum steering committee see: www.osemosys.org]
Licensed under the Apache License, Version 2.0

Estructura del archivo:
  - Sets: YEAR, TECHNOLOGY, TIMESLICE, FUEL, EMISSION, MODE_OF_OPERATION, REGION; opcionales STORAGE, SEASON, DAYTYPE, DAILYTIMEBRACKET, UDC.
  - Parameters: demanda, rendimientos (CapacityFactor, ActivityRatios), costes, límites de capacidad/actividad, emisiones, UDC, almacenamiento.
  - Variables: NewCapacity, RateOfActivity, costes descontados, emisiones, reserva, RE, almacenamiento.
  - Objective: minimizar suma de TotalDiscountedCost por región y año.
  - Constraints: capacidad, balance energético, costes, límites, emisiones, reserve margin, UDC, almacenamiento.
"""

from __future__ import annotations

from pyomo.environ import (
    AbstractModel,
    Constraint,
    NonNegativeIntegers,
    NonNegativeReals,
    Objective,
    Param,
    Reals,
    Set,
    Var,
    minimize,
)


def create_abstract_model(
    has_storage: bool = False,
    has_udc: bool = True,
) -> AbstractModel:
    """Construye el AbstractModel OSeMOSYS completo.

    Parameters
    ----------
    has_storage : bool
        Si True, incluye sets/params/vars/constraints de almacenamiento.
    has_udc : bool
        Si True, incluye User-Defined Constraints.

    Returns
    -------
    AbstractModel listo para ``model.create_instance(data)``.
    """
    model = AbstractModel()

    # ====================================================================
    #    Sets (conjuntos que indexan parámetros y variables)
    # ====================================================================

    model.YEAR = Set(ordered=True)
    model.TECHNOLOGY = Set()
    model.TIMESLICE = Set(ordered=True)
    model.FUEL = Set()
    model.EMISSION = Set()
    model.MODE_OF_OPERATION = Set()
    model.REGION = Set()

    if has_storage:
        model.STORAGE = Set()
        model.SEASON = Set()
        model.DAYTYPE = Set()
        model.DAILYTIMEBRACKET = Set()
        model.STORAGEINTRADAY = Set()
        model.STORAGEINTRAYEAR = Set()

    model.FLEXIBLEDEMANDTYPE = Set()
    model.UDC = Set()

    # ====================================================================
    #    Parameters — Global (YearSplit, descuento, vida operativa, factores de recuperación)
    # ====================================================================

    model.YearSplit = Param(model.TIMESLICE, model.YEAR)
    model.DiscountRate = Param(model.REGION, default=0.05)

    def DiscountRateIdv_init(m, r, t):
        return m.DiscountRate[r]
    model.DiscountRateIdv = Param(
        model.REGION, model.TECHNOLOGY,
        within=NonNegativeReals, initialize=DiscountRateIdv_init, mutable=True,
    )
    model.OperationalLife = Param(model.REGION, model.TECHNOLOGY, default=1)

    def CapitalRecoveryFactor_rule(m, r, t):
        dr = m.DiscountRateIdv[r, t]
        ol = m.OperationalLife[r, t]
        return (1 - (1 + dr)**(-1)) / (1 - (1 + dr)**(-ol))
    model.CapitalRecoveryFactor = Param(
        model.REGION, model.TECHNOLOGY,
        initialize=CapitalRecoveryFactor_rule, within=Reals, mutable=True,
    )

    def PvAnnuity_rule(m, r, t):
        dr = m.DiscountRate[r]
        ol = m.OperationalLife[r, t]
        return (1 - (1 + dr)**(-ol)) * (1 + dr) / dr
    model.PvAnnuity = Param(
        model.REGION, model.TECHNOLOGY,
        initialize=PvAnnuity_rule, within=Reals, mutable=True,
    )

    model.DepreciationMethod = Param(model.REGION, default=1)

    # ====================================================================
    #    Parameters — Demands
    # ====================================================================

    model.AccumulatedAnnualDemand = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.SpecifiedAnnualDemand = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.SpecifiedDemandProfile = Param(
        model.REGION, model.FUEL, model.TIMESLICE, model.YEAR, default=0,
    )

    def Demand_init(m, r, l, f, y):
        if m.SpecifiedAnnualDemand[r, f, y] > 0:
            return m.SpecifiedAnnualDemand[r, f, y] * m.SpecifiedDemandProfile[r, f, l, y]
        return 0.0
    model.Demand = Param(
        model.REGION, model.TIMESLICE, model.FUEL, model.YEAR,
        initialize=Demand_init, default=0.0,
    )

    # ====================================================================
    #    Parameters — Performance
    # ====================================================================

    model.CapacityToActivityUnit = Param(model.REGION, model.TECHNOLOGY, default=1)
    model.CapacityFactor = Param(
        model.REGION, model.TECHNOLOGY, model.TIMESLICE, model.YEAR, default=1,
    )
    model.AvailabilityFactor = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=1)
    model.ResidualCapacity = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0)
    model.InputActivityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.MODE_OF_OPERATION, model.YEAR,
        default=0,
    )
    model.OutputActivityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.MODE_OF_OPERATION, model.YEAR,
        default=0,
    )

    # ====================================================================
    #    Parameters — Technology Costs
    # ====================================================================

    model.CapitalCost = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0.000001)
    model.VariableCost = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        default=0.000001,
    )
    model.FixedCost = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0)

    # ====================================================================
    #    Parameters — Capacity Constraints
    # ====================================================================

    model.CapacityOfOneTechnologyUnit = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )
    model.TotalAnnualMaxCapacity = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=9999999,
    )
    model.TotalAnnualMinCapacity = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )

    # ====================================================================
    #    Parameters — Investment Constraints
    # ====================================================================

    model.TotalAnnualMaxCapacityInvestment = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=9999999,
    )
    model.TotalAnnualMinCapacityInvestment = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )

    # ====================================================================
    #    Parameters — Activity Constraints
    # ====================================================================

    model.TotalTechnologyAnnualActivityUpperLimit = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=9999999,
    )
    model.TotalTechnologyAnnualActivityLowerLimit = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )
    model.TotalTechnologyModelPeriodActivityUpperLimit = Param(
        model.REGION, model.TECHNOLOGY, default=9999999,
    )
    model.TotalTechnologyModelPeriodActivityLowerLimit = Param(
        model.REGION, model.TECHNOLOGY, default=0,
    )

    # ====================================================================
    #    Parameters — Reserve Margin
    # ====================================================================

    model.ReserveMarginTagTechnology = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )
    model.ReserveMarginTagFuel = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.ReserveMargin = Param(model.REGION, model.YEAR, default=1)

    # ====================================================================
    #    Parameters — RE Generation Target
    # ====================================================================

    model.RETagTechnology = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0)
    model.RETagFuel = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.REMinProductionTarget = Param(model.REGION, model.YEAR, default=0)

    # ====================================================================
    #    Parameters — Emissions & Penalties
    # ====================================================================

    model.EmissionActivityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        default=0,
    )
    model.EmissionsPenalty = Param(model.REGION, model.EMISSION, model.YEAR, default=0)
    model.AnnualExogenousEmission = Param(
        model.REGION, model.EMISSION, model.YEAR, default=0,
    )
    model.AnnualEmissionLimit = Param(
        model.REGION, model.EMISSION, model.YEAR, default=9999999,
    )
    model.ModelPeriodExogenousEmission = Param(model.REGION, model.EMISSION, default=0)
    model.ModelPeriodEmissionLimit = Param(model.REGION, model.EMISSION, default=9999999)

    # ====================================================================
    #    Parameters — MUIO
    # ====================================================================

    model.InputToNewCapacityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.YEAR, within=Reals, default=0,
    )
    model.InputToTotalCapacityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.YEAR, within=Reals, default=0,
    )
    model.TechnologyActivityByModeLowerLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.TechnologyActivityByModeUpperLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.TechnologyActivityDecreaseByModeLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.TechnologyActivityIncreaseByModeLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.EmissionToActivityChangeRatio = Param(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )

    # ====================================================================
    #    Parameters — UDC (User-Defined Constraints)
    # ====================================================================

    if has_udc:
        model.UDCMultiplierTotalCapacity = Param(
            model.REGION, model.TECHNOLOGY, model.UDC, model.YEAR,
            within=Reals, default=0,
        )
        model.UDCMultiplierNewCapacity = Param(
            model.REGION, model.TECHNOLOGY, model.UDC, model.YEAR,
            within=Reals, default=0,
        )
        model.UDCMultiplierActivity = Param(
            model.REGION, model.TECHNOLOGY, model.UDC, model.YEAR,
            within=Reals, default=0,
        )
        model.UDCConstant = Param(
            model.REGION, model.UDC, model.YEAR, within=Reals, default=0,
        )
        model.UDCTag = Param(model.REGION, model.UDC, within=Reals, default=2)

    # ====================================================================
    #    Parameters — Storage
    # ====================================================================

    if has_storage:
        model.DaySplit = Param(model.DAILYTIMEBRACKET, model.YEAR, default=0.00137)
        model.Conversionls = Param(model.TIMESLICE, model.SEASON, default=0)
        model.Conversionld = Param(model.TIMESLICE, model.DAYTYPE, default=0)
        model.Conversionlh = Param(model.TIMESLICE, model.DAILYTIMEBRACKET, default=0)
        model.DaysInDayType = Param(model.SEASON, model.DAYTYPE, model.YEAR, default=7)
        model.TechnologyToStorage = Param(
            model.REGION, model.TECHNOLOGY, model.STORAGE, model.MODE_OF_OPERATION, default=0,
        )
        model.TechnologyFromStorage = Param(
            model.REGION, model.TECHNOLOGY, model.STORAGE, model.MODE_OF_OPERATION, default=0,
        )
        model.StorageLevelStart = Param(model.REGION, model.STORAGE, default=0.0000001)
        model.StorageMaxChargeRate = Param(model.REGION, model.STORAGE, default=9999999)
        model.StorageMaxDischargeRate = Param(model.REGION, model.STORAGE, default=9999999)
        model.MinStorageCharge = Param(model.REGION, model.STORAGE, model.YEAR, default=0)
        model.OperationalLifeStorage = Param(model.REGION, model.STORAGE, default=0)
        model.CapitalCostStorage = Param(
            model.REGION, model.STORAGE, model.YEAR, default=0,
        )
        model.ResidualStorageCapacity = Param(
            model.REGION, model.STORAGE, model.YEAR, default=0,
        )

    # ====================================================================
    #    Parameters — Disposal / Recovery (Max B/C)
    # ====================================================================

    model.DisposalCostPerCapacity = Param(model.REGION, model.TECHNOLOGY, default=0.0)
    model.RecoveryValuePerCapacity = Param(model.REGION, model.TECHNOLOGY, default=0.0)

    # ====================================================================
    #    Variables — Capacity
    # ====================================================================

    model.NumberOfNewTechnologyUnits = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeIntegers, initialize=0,
    )
    model.NewCapacity = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Activity
    # ====================================================================

    model.RateOfActivity = Var(
        model.REGION, model.TIMESLICE, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Costing
    # ====================================================================

    model.VariableOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.TIMESLICE, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.SalvageValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedSalvageValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.OperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.CapitalInvestment = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedCapitalInvestment = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.AnnualVariableOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.AnnualFixedOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.TotalDiscountedCostByTechnology = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.TotalDiscountedCost = Var(
        model.REGION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Reserve Margin
    # ====================================================================

    model.TotalCapacityInReserveMargin = Var(
        model.REGION, model.YEAR, domain=NonNegativeReals, initialize=0.0,
    )
    model.DemandNeedingReserveMargin = Var(
        model.REGION, model.TIMESLICE, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — RE Gen Target
    # ====================================================================

    model.TotalREProductionAnnual = Var(model.REGION, model.YEAR, initialize=0.0)
    model.RETotalProductionOfTargetFuelAnnual = Var(
        model.REGION, model.YEAR, initialize=0.0,
    )
    model.TotalTechnologyModelPeriodActivity = Var(
        model.REGION, model.TECHNOLOGY, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Emissions
    # ====================================================================

    model.AnnualTechnologyEmissionByMode = Var(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualTechnologyEmission = Var(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualTechnologyEmissionPenaltyByEmission = Var(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualTechnologyEmissionsPenalty = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedTechnologyEmissionsPenalty = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualEmissions = Var(
        model.REGION, model.EMISSION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.ModelPeriodEmissions = Var(
        model.REGION, model.EMISSION,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Storage
    # ====================================================================

    if has_storage:
        model.NewStorageCapacity = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.SalvageValueStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelYearStart = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelYearFinish = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.RateOfStorageCharge = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.RateOfStorageDischarge = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.NetChargeWithinYear = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=Reals, initialize=0.0,
        )
        model.NetChargeWithinDay = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=Reals, initialize=0.0,
        )
        model.StorageLevelSeasonStart = Var(
            model.REGION, model.STORAGE, model.SEASON, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelDayTypeStart = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelDayTypeFinish = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLowerLimit = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageUpperLimit = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.AccumulatedNewStorageCapacity = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.CapitalInvestmentStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.DiscountedCapitalInvestmentStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.DiscountedSalvageValueStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.TotalDiscountedStorageCost = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )

    # ====================================================================
    #    Variables — Disposal / Recovery
    # ====================================================================

    model.DisposalCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedDisposalCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.RecoveryValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedRecoveryValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ####################################################################
    #    Objective Function
    # ####################################################################

    def ObjectiveFunction_rule(m):
        return sum(m.TotalDiscountedCost[r, y] for r in m.REGION for y in m.YEAR)
    model.OBJ = Objective(rule=ObjectiveFunction_rule, sense=minimize)

    # ####################################################################
    #    Constraints — Capacity Adequacy A
    # ####################################################################

    def TotalNewCapacity_2_rule(m, r, t, y):
        if m.CapacityOfOneTechnologyUnit[r, t, y] != 0:
            return (
                m.CapacityOfOneTechnologyUnit[r, t, y]
                * m.NumberOfNewTechnologyUnits[r, t, y]
                == m.NewCapacity[r, t, y]
            )
        return Constraint.Skip
    model.TotalNewCapacity_2 = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalNewCapacity_2_rule,
    )

    # ConstraintCapacity: en cada (r,l,t,y) la suma de RateOfActivity por modo <=
    # (capacidad nueva acumulada + residual) * CapacityFactor * CapacityToActivityUnit.
    def ConstraintCapacity_rule(m, r, l, t, y):
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
            <= (
                sum(
                    m.NewCapacity[r, t, yy]
                    for yy in m.YEAR
                    if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                )
                + m.ResidualCapacity[r, t, y]
            )
            * m.CapacityFactor[r, t, l, y]
            * m.CapacityToActivityUnit[r, t]
        )
    model.ConstraintCapacity = Constraint(
        model.REGION, model.TIMESLICE, model.TECHNOLOGY, model.YEAR,
        rule=ConstraintCapacity_rule,
    )

    # ####################################################################
    #    Constraints — Capacity Adequacy B
    # ####################################################################

    def PlannedMaintenance_rule(m, r, t, y):
        return (
            sum(
                m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                for l in m.TIMESLICE
                for mo in m.MODE_OF_OPERATION
            )
            <= sum(
                (
                    sum(
                        m.NewCapacity[r, t, yy]
                        for yy in m.YEAR
                        if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                    )
                    + m.ResidualCapacity[r, t, y]
                )
                * m.CapacityFactor[r, t, l, y]
                * m.YearSplit[l, y]
                for l in m.TIMESLICE
            )
            * m.AvailabilityFactor[r, t, y]
            * m.CapacityToActivityUnit[r, t]
        )
    model.PlannedMaintenance = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=PlannedMaintenance_rule,
    )

    # ####################################################################
    #    Constraints — Energy Balance A (por timeslice: oferta >= demanda + inputs)
    # ####################################################################

    def EnergyBalanceEachTS5_rule(m, r, l, f, y):
        return (
            sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.OutputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for mo in m.MODE_OF_OPERATION
                for t in m.TECHNOLOGY
            )
            >= m.Demand[r, l, f, y]
            + sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.InputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for mo in m.MODE_OF_OPERATION
                for t in m.TECHNOLOGY
            )
        )
    model.EnergyBalanceEachTS5 = Constraint(
        model.REGION, model.TIMESLICE, model.FUEL, model.YEAR,
        rule=EnergyBalanceEachTS5_rule,
    )

    # ####################################################################
    #    Constraints — Energy Balance B (AccumulatedAnnualDemand)
    # ####################################################################

    def EnergyBalanceEachYear4_rule(m, r, f, y):
        return (
            sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.OutputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for t in m.TECHNOLOGY
                for mo in m.MODE_OF_OPERATION
                for l in m.TIMESLICE
            )
            >= sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.InputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for mo in m.MODE_OF_OPERATION
                for t in m.TECHNOLOGY
                for l in m.TIMESLICE
            )
            + m.AccumulatedAnnualDemand[r, f, y]
        )
    model.EnergyBalanceEachYear4 = Constraint(
        model.REGION, model.FUEL, model.YEAR,
        rule=EnergyBalanceEachYear4_rule,
    )

    # ####################################################################
    #    Constraints — Capital Costs
    # ####################################################################

    def UndiscountedCapitalInvestment_rule(m, r, t, y):
        return (
            m.CapitalCost[r, t, y]
            * m.NewCapacity[r, t, y]
            * m.CapitalRecoveryFactor[r, t]
            * m.PvAnnuity[r, t]
            == m.CapitalInvestment[r, t, y]
        )
    model.UndiscountedCapitalInvestment = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=UndiscountedCapitalInvestment_rule,
    )

    def DiscountedCapitalInvestment_rule(m, r, t, y):
        return (
            m.CapitalInvestment[r, t, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR)))
            == m.DiscountedCapitalInvestment[r, t, y]
        )
    model.DiscountedCapitalInvestment_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DiscountedCapitalInvestment_rule,
    )

    # ####################################################################
    #    Constraints — Operating Costs
    # ####################################################################

    def OperatingCostsVariable_rule(m, r, t, y):
        return (
            sum(
                sum(
                    m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                    for l in m.TIMESLICE
                )
                * m.VariableCost[r, t, mo, y]
                for mo in m.MODE_OF_OPERATION
            )
            == m.AnnualVariableOperatingCost[r, t, y]
        )
    model.OperatingCostsVariable = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=OperatingCostsVariable_rule,
    )

    def OperatingCostsFixedAnnual_rule(m, r, t, y):
        return (
            (
                sum(
                    m.NewCapacity[r, t, yy]
                    for yy in m.YEAR
                    if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                )
                + m.ResidualCapacity[r, t, y]
            )
            * m.FixedCost[r, t, y]
            == m.AnnualFixedOperatingCost[r, t, y]
        )
    model.OperatingCostsFixedAnnual = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=OperatingCostsFixedAnnual_rule,
    )

    def OperatingCostsTotalAnnual_rule(m, r, t, y):
        return (
            m.AnnualFixedOperatingCost[r, t, y]
            + m.AnnualVariableOperatingCost[r, t, y]
            == m.OperatingCost[r, t, y]
        )
    model.OperatingCostsTotalAnnual = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=OperatingCostsTotalAnnual_rule,
    )

    def DiscountedOperatingCostsTotalAnnual_rule(m, r, t, y):
        return (
            m.OperatingCost[r, t, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR) + 0.5))
            == m.DiscountedOperatingCost[r, t, y]
        )
    model.DiscountedOperatingCostsTotalAnnual = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DiscountedOperatingCostsTotalAnnual_rule,
    )

    # ####################################################################
    #    Constraints — Total Discounted Costs
    # ####################################################################

    def TotalDiscountedCostByTechnology_rule(m, r, t, y):
        return (
            m.DiscountedOperatingCost[r, t, y]
            + m.DiscountedCapitalInvestment[r, t, y]
            + m.DiscountedTechnologyEmissionsPenalty[r, t, y]
            - m.DiscountedSalvageValue[r, t, y]
            + m.DiscountedDisposalCost[r, t, y]
            - m.DiscountedRecoveryValue[r, t, y]
            == m.TotalDiscountedCostByTechnology[r, t, y]
        )
    model.TotalDiscountedCostByTechnology_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalDiscountedCostByTechnology_rule,
    )

    if has_storage:
        def TotalDiscountedCost_rule(m, r, y):
            return (
                sum(m.TotalDiscountedCostByTechnology[r, t, y] for t in m.TECHNOLOGY)
                + sum(m.TotalDiscountedStorageCost[r, s, y] for s in m.STORAGE)
                == m.TotalDiscountedCost[r, y]
            )
        model.TotalDiscountedCost_constraint = Constraint(
            model.REGION, model.YEAR, rule=TotalDiscountedCost_rule,
        )
    else:
        def TotalDiscountedCost_rule(m, r, y):
            return (
                sum(m.TotalDiscountedCostByTechnology[r, t, y] for t in m.TECHNOLOGY)
                == m.TotalDiscountedCost[r, y]
            )
        model.TotalDiscountedCost_constraint = Constraint(
            model.REGION, model.YEAR, rule=TotalDiscountedCost_rule,
        )

    # ####################################################################
    #    Constraints — Total Capacity Constraints
    # ####################################################################

    def TotalAnnualMaxCapacityConstraint_rule(m, r, t, y):
        if m.TotalAnnualMaxCapacity[r, t, y] == 9999999:
            return Constraint.Skip
        return (
            sum(
                m.NewCapacity[r, t, yy]
                for yy in m.YEAR
                if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
            )
            + m.ResidualCapacity[r, t, y]
            <= m.TotalAnnualMaxCapacity[r, t, y]
        )
    model.TotalAnnualMaxCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMaxCapacityConstraint_rule,
    )

    def TotalAnnualMinCapacityConstraint_rule(m, r, t, y):
        return (
            sum(
                m.NewCapacity[r, t, yy]
                for yy in m.YEAR
                if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
            )
            + m.ResidualCapacity[r, t, y]
            >= m.TotalAnnualMinCapacity[r, t, y]
        )
    model.TotalAnnualMinCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMinCapacityConstraint_rule,
    )

    # ####################################################################
    #    Constraints — Salvage Value
    # ####################################################################

    def SalvageValueAtEndOfPeriod1_rule(m, r, t, y):
        if (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLife[r, t] - 1) > max(m.YEAR))
            and m.DiscountRate[r] > 0
        ):
            return m.SalvageValue[r, t, y] == (
                m.CapitalCost[r, t, y]
                * m.NewCapacity[r, t, y]
                * m.CapitalRecoveryFactor[r, t]
                * m.PvAnnuity[r, t]
                * (
                    1
                    - (
                        ((1 + m.DiscountRate[r]) ** (max(m.YEAR) - y + 1) - 1)
                        / ((1 + m.DiscountRate[r]) ** m.OperationalLife[r, t] - 1)
                    )
                )
            )
        elif (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLife[r, t] - 1) > max(m.YEAR))
            and m.DiscountRate[r] == 0
        ) or (
            m.DepreciationMethod[r] == 2
            and (y + m.OperationalLife[r, t] - 1) > max(m.YEAR)
        ):
            return m.SalvageValue[r, t, y] == (
                m.CapitalCost[r, t, y]
                * m.NewCapacity[r, t, y]
                * m.CapitalRecoveryFactor[r, t]
                * m.PvAnnuity[r, t]
                * (1 - (max(m.YEAR) - y + 1) / m.OperationalLife[r, t])
            )
        return m.SalvageValue[r, t, y] == 0
    model.SalvageValueAtEndOfPeriod1 = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=SalvageValueAtEndOfPeriod1_rule,
    )

    def SalvageValueDiscountedToStartYear_rule(m, r, t, y):
        return m.DiscountedSalvageValue[r, t, y] == m.SalvageValue[r, t, y] / (
            (1 + m.DiscountRate[r]) ** (1 + max(m.YEAR) - min(m.YEAR))
        )
    model.SalvageValueDiscountedToStartYear = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=SalvageValueDiscountedToStartYear_rule,
    )

    # ####################################################################
    #    Constraints — New Capacity Constraints
    # ####################################################################

    def TotalAnnualMaxNewCapacityConstraint_rule(m, r, t, y):
        if m.TotalAnnualMaxCapacityInvestment[r, t, y] == 9999999:
            return Constraint.Skip
        return m.NewCapacity[r, t, y] <= m.TotalAnnualMaxCapacityInvestment[r, t, y]
    model.TotalAnnualMaxNewCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMaxNewCapacityConstraint_rule,
    )

    def TotalAnnualMinNewCapacityConstraint_rule(m, r, t, y):
        return m.NewCapacity[r, t, y] >= m.TotalAnnualMinCapacityInvestment[r, t, y]
    model.TotalAnnualMinNewCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMinNewCapacityConstraint_rule,
    )

    # ####################################################################
    #    Constraints — Annual Activity Constraints
    # ####################################################################

    def TotalAnnualTechnologyActivityUpperLimit_rule(m, r, t, y):
        if m.TotalTechnologyAnnualActivityUpperLimit[r, t, y] == 9999999:
            return Constraint.Skip
        return (
            sum(
                sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
                * m.YearSplit[l, y]
                for l in m.TIMESLICE
            )
            <= m.TotalTechnologyAnnualActivityUpperLimit[r, t, y]
        )
    model.TotalAnnualTechnologyActivityUpperlimit = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualTechnologyActivityUpperLimit_rule,
    )

    def TotalAnnualTechnologyActivityLowerLimit_rule(m, r, t, y):
        return (
            sum(
                sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
                * m.YearSplit[l, y]
                for l in m.TIMESLICE
            )
            >= m.TotalTechnologyAnnualActivityLowerLimit[r, t, y]
        )
    model.TotalAnnualTechnologyActivityLowerlimit = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualTechnologyActivityLowerLimit_rule,
    )

    # ####################################################################
    #    Constraints — Total Activity Constraints
    # ####################################################################

    def TotalModelHorizonTechnologyActivity_rule(m, r, t):
        return (
            sum(
                sum(
                    sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
                    * m.YearSplit[l, y]
                    for l in m.TIMESLICE
                )
                for y in m.YEAR
            )
            == m.TotalTechnologyModelPeriodActivity[r, t]
        )
    model.TotalModelHorizonTechnologyActivity = Constraint(
        model.REGION, model.TECHNOLOGY,
        rule=TotalModelHorizonTechnologyActivity_rule,
    )

    def TotalModelHorizonTechnologyActivityUpperLimit_rule(m, r, t):
        if m.TotalTechnologyModelPeriodActivityUpperLimit[r, t] == 9999999:
            return Constraint.Skip
        return (
            m.TotalTechnologyModelPeriodActivity[r, t]
            <= m.TotalTechnologyModelPeriodActivityUpperLimit[r, t]
        )
    model.TotalModelHorizonTechnologyActivityUpperLimit = Constraint(
        model.REGION, model.TECHNOLOGY,
        rule=TotalModelHorizonTechnologyActivityUpperLimit_rule,
    )

    def TotalModelHorizonTechnologyActivityLowerLimit_rule(m, r, t):
        return (
            m.TotalTechnologyModelPeriodActivity[r, t]
            >= m.TotalTechnologyModelPeriodActivityLowerLimit[r, t]
        )
    model.TotalModelHorizonTechnologyActivityLowerLimit = Constraint(
        model.REGION, model.TECHNOLOGY,
        rule=TotalModelHorizonTechnologyActivityLowerLimit_rule,
    )

    # ####################################################################
    #    Constraints — Emissions Accounting (emisión por tecnología/modo, límites anuales y periodo)
    # ####################################################################

    def AnnualEmissionProductionByMode_rule(m, r, t, e, mo, y):
        if m.EmissionActivityRatio[r, t, e, mo, y] != 0:
            return (
                m.EmissionActivityRatio[r, t, e, mo, y]
                * sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
                == m.AnnualTechnologyEmissionByMode[r, t, e, mo, y]
            )
        return m.AnnualTechnologyEmissionByMode[r, t, e, mo, y] == 0
    model.AnnualEmissionProductionByMode = Constraint(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        rule=AnnualEmissionProductionByMode_rule,
    )

    def AnnualEmissionProduction_rule(m, r, t, e, y):
        return (
            sum(m.AnnualTechnologyEmissionByMode[r, t, e, mo, y] for mo in m.MODE_OF_OPERATION)
            == m.AnnualTechnologyEmission[r, t, e, y]
        )
    model.AnnualEmissionProduction = Constraint(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        rule=AnnualEmissionProduction_rule,
    )

    def EmissionPenaltyByTechAndEmission_rule(m, r, t, e, y):
        return (
            m.AnnualTechnologyEmission[r, t, e, y] * m.EmissionsPenalty[r, e, y]
            == m.AnnualTechnologyEmissionPenaltyByEmission[r, t, e, y]
        )
    model.EmissionPenaltyByTechAndEmission = Constraint(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        rule=EmissionPenaltyByTechAndEmission_rule,
    )

    def EmissionsPenaltyByTechnology_rule(m, r, t, y):
        return (
            sum(m.AnnualTechnologyEmissionPenaltyByEmission[r, t, e, y] for e in m.EMISSION)
            == m.AnnualTechnologyEmissionsPenalty[r, t, y]
        )
    model.EmissionsPenaltyByTechnology = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=EmissionsPenaltyByTechnology_rule,
    )

    def DiscountedEmissionsPenaltyByTechnology_rule(m, r, t, y):
        return (
            m.AnnualTechnologyEmissionsPenalty[r, t, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR) + 0.5))
            == m.DiscountedTechnologyEmissionsPenalty[r, t, y]
        )
    model.DiscountedEmissionsPenaltyByTechnology = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DiscountedEmissionsPenaltyByTechnology_rule,
    )

    def EmissionsAccounting1_rule(m, r, e, y):
        return (
            sum(m.AnnualTechnologyEmission[r, t, e, y] for t in m.TECHNOLOGY)
            == m.AnnualEmissions[r, e, y]
        )
    model.EmissionsAccounting1 = Constraint(
        model.REGION, model.EMISSION, model.YEAR,
        rule=EmissionsAccounting1_rule,
    )

    def EmissionsAccounting2_rule(m, r, e):
        return (
            sum(m.AnnualEmissions[r, e, y] for y in m.YEAR)
            == m.ModelPeriodEmissions[r, e] - m.ModelPeriodExogenousEmission[r, e]
        )
    model.EmissionsAccounting2 = Constraint(
        model.REGION, model.EMISSION,
        rule=EmissionsAccounting2_rule,
    )

    def AnnualEmissionsLimit_rule(m, r, e, y):
        if m.AnnualEmissionLimit[r, e, y] == 9999999:
            return Constraint.Skip
        return (
            m.AnnualEmissions[r, e, y] + m.AnnualExogenousEmission[r, e, y]
            <= m.AnnualEmissionLimit[r, e, y]
        )
    model.AnnualEmissionsLimit = Constraint(
        model.REGION, model.EMISSION, model.YEAR,
        rule=AnnualEmissionsLimit_rule,
    )

    def ModelPeriodEmissionsLimit_rule(m, r, e):
        if m.ModelPeriodEmissionLimit[r, e] == 9999999:
            return Constraint.Skip
        return m.ModelPeriodEmissions[r, e] <= m.ModelPeriodEmissionLimit[r, e]
    model.ModelPeriodEmissionsLimit = Constraint(
        model.REGION, model.EMISSION,
        rule=ModelPeriodEmissionsLimit_rule,
    )

    # ####################################################################
    #    Constraints — Reserve Margin (capacidad en reserva >= demanda * factor por timeslice)
    # ####################################################################

    def ReserveMargin_TechnologiesIncluded_rule(m, r, y):
        return (
            sum(
                (
                    sum(
                        m.NewCapacity[r, t, yy]
                        for yy in m.YEAR
                        if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                    )
                    + m.ResidualCapacity[r, t, y]
                )
                * m.ReserveMarginTagTechnology[r, t, y]
                * m.CapacityToActivityUnit[r, t]
                for t in m.TECHNOLOGY
            )
            == m.TotalCapacityInReserveMargin[r, y]
        )
    model.ReserveMargin_TechnologiesIncluded = Constraint(
        model.REGION, model.YEAR,
        rule=ReserveMargin_TechnologiesIncluded_rule,
    )

    def ReserveMargin_FuelsIncluded_rule(m, r, l, y):
        return (
            sum(
                sum(
                    m.RateOfActivity[r, l, t, mo, y] * m.OutputActivityRatio[r, t, f, mo, y]
                    for t in m.TECHNOLOGY
                    for mo in m.MODE_OF_OPERATION
                )
                * m.ReserveMarginTagFuel[r, f, y]
                for f in m.FUEL
            )
            == m.DemandNeedingReserveMargin[r, l, y]
        )
    model.ReserveMargin_FuelsIncluded = Constraint(
        model.REGION, model.TIMESLICE, model.YEAR,
        rule=ReserveMargin_FuelsIncluded_rule,
    )

    def ReserveMarginConstraint_rule(m, r, l, y):
        return (
            m.DemandNeedingReserveMargin[r, l, y] * m.ReserveMargin[r, y]
            <= m.TotalCapacityInReserveMargin[r, y]
        )
    model.ReserveMarginConstraint = Constraint(
        model.REGION, model.TIMESLICE, model.YEAR,
        rule=ReserveMarginConstraint_rule,
    )

    # ####################################################################
    #    Constraints — MUIO
    # ####################################################################

    def LU1_rule(m, r, t, mo, y):
        if m.TechnologyActivityByModeUpperLimit[r, t, mo, y] == 0:
            return Constraint.Skip
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            <= m.TechnologyActivityByModeUpperLimit[r, t, mo, y]
        )
    model.LU1_TechnologyActivityByModeUL = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        rule=LU1_rule,
    )

    def LU2_rule(m, r, t, mo, y):
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            >= m.TechnologyActivityByModeLowerLimit[r, t, mo, y]
        )
    model.LU2_TechnologyActivityByModeLL = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        rule=LU2_rule,
    )

    def LU3_rule(m, r, t, mo, y, yy):
        if (y - yy != 1) or (m.TechnologyActivityIncreaseByModeLimit[r, t, mo, yy] == 0):
            return Constraint.Skip
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            <= (1 + m.TechnologyActivityIncreaseByModeLimit[r, t, mo, yy])
            * sum(m.RateOfActivity[r, l, t, mo, yy] * m.YearSplit[l, yy] for l in m.TIMESLICE)
        )
    model.LU3_TechnologyActivityIncreaseByMode = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR, model.YEAR,
        rule=LU3_rule,
    )

    def LU4_rule(m, r, t, mo, y, yy):
        if (y - yy != 1) or (m.TechnologyActivityDecreaseByModeLimit[r, t, mo, yy] == 0):
            return Constraint.Skip
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            >= (1 - m.TechnologyActivityDecreaseByModeLimit[r, t, mo, yy])
            * sum(m.RateOfActivity[r, l, t, mo, yy] * m.YearSplit[l, yy] for l in m.TIMESLICE)
        )
    model.LU4_TechnologyActivityDecreaseByMode = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR, model.YEAR,
        rule=LU4_rule,
    )

    # ####################################################################
    #    Constraints — UDC (User-Defined Constraints: combinación lineal de capacidad/actividad vs constante)
    # ####################################################################

    if has_udc:
        def UDC1_rule(m, r, u, y):
            if m.UDCTag[r, u] != 0:
                return Constraint.Skip
            return (
                sum(
                    m.UDCMultiplierTotalCapacity[r, t, u, y]
                    * (
                        sum(
                            m.NewCapacity[r, t, yy]
                            for yy in m.YEAR
                            if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                        )
                        + m.ResidualCapacity[r, t, y]
                    )
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierNewCapacity[r, t, u, y] * m.NewCapacity[r, t, y]
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierActivity[r, t, u, y]
                    * sum(
                        m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                        for l in m.TIMESLICE
                        for mo in m.MODE_OF_OPERATION
                    )
                    for t in m.TECHNOLOGY
                )
                <= m.UDCConstant[r, u, y]
            )
        model.UDC1_UserDefinedConstraintInequality = Constraint(
            model.REGION, model.UDC, model.YEAR, rule=UDC1_rule,
        )

        def UDC2_rule(m, r, u, y):
            if m.UDCTag[r, u] != 1:
                return Constraint.Skip
            return (
                sum(
                    m.UDCMultiplierTotalCapacity[r, t, u, y]
                    * (
                        sum(
                            m.NewCapacity[r, t, yy]
                            for yy in m.YEAR
                            if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                        )
                        + m.ResidualCapacity[r, t, y]
                    )
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierNewCapacity[r, t, u, y] * m.NewCapacity[r, t, y]
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierActivity[r, t, u, y]
                    * sum(
                        m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                        for l in m.TIMESLICE
                        for mo in m.MODE_OF_OPERATION
                    )
                    for t in m.TECHNOLOGY
                )
                == m.UDCConstant[r, u, y]
            )
        model.UDC2_UserDefinedConstraintEquality = Constraint(
            model.REGION, model.UDC, model.YEAR, rule=UDC2_rule,
        )

    # ####################################################################
    #    Constraints — Disposal / Recovery (Max B/C)
    # ####################################################################

    def DisposalCostAtEndOfPeriod1_rule(m, r, t, y):
        return m.DisposalCost[r, t, y] == (
            m.DisposalCostPerCapacity[r, t] * m.CapitalCost[r, t, y] * m.NewCapacity[r, t, y]
        )
    model.DisposalCostAtEndOfPeriod1 = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DisposalCostAtEndOfPeriod1_rule,
    )

    def DisposalCostDiscounted_rule(m, r, t, y):
        return m.DiscountedDisposalCost[r, t, y] == m.DisposalCost[r, t, y] / (
            (1 + m.DiscountRate[r]) ** (y + m.OperationalLife[r, t] - min(m.YEAR))
        )
    model.DiscountedDisposalCost_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DisposalCostDiscounted_rule,
    )

    def RecoveryValueAtEndOfPeriod_rule(m, r, t, y):
        return m.RecoveryValue[r, t, y] == (
            m.RecoveryValuePerCapacity[r, t] * m.CapitalCost[r, t, y] * m.NewCapacity[r, t, y]
        )
    model.RecoveryValueAtEndOfPeriod = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=RecoveryValueAtEndOfPeriod_rule,
    )

    def RecoveryValueDiscounted_rule(m, r, t, y):
        return m.DiscountedRecoveryValue[r, t, y] == m.RecoveryValue[r, t, y] / (
            (1 + m.DiscountRate[r]) ** (y + m.OperationalLife[r, t] - min(m.YEAR))
        )
    model.RecoveryValueDiscounted_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=RecoveryValueDiscounted_rule,
    )

    # ####################################################################
    #    Constraints — Storage
    # ####################################################################

    if has_storage:
        _add_storage_constraints(model)

    return model


# ========================================================================
#  Storage constraints (separadas en helper para legibilidad)
# ========================================================================

def _add_storage_constraints(model: AbstractModel) -> None:
    """Agrega todas las restricciones de almacenamiento al modelo."""

    # --- Storage equations ---

    def RateOfStorageCharge_rule(m, r, s, ls, ld, lh, y, t, mo):
        if m.TechnologyToStorage[r, t, s, mo] > 0:
            return (
                sum(
                    m.RateOfActivity[r, l, t, mo, y]
                    * m.TechnologyToStorage[r, t, s, mo]
                    * m.Conversionls[l, ls]
                    * m.Conversionld[l, ld]
                    * m.Conversionlh[l, lh]
                    for mo in m.MODE_OF_OPERATION
                    for l in m.TIMESLICE
                    for t in m.TECHNOLOGY
                )
                == m.RateOfStorageCharge[r, s, ls, ld, lh, y]
            )
        return Constraint.Skip
    model.RateOfStorageCharge_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR, model.TECHNOLOGY, model.MODE_OF_OPERATION,
        rule=RateOfStorageCharge_rule,
    )

    def RateOfStorageDischarge_rule(m, r, s, ls, ld, lh, y, t, mo):
        if m.TechnologyFromStorage[r, t, s, mo] > 0:
            return (
                sum(
                    m.RateOfActivity[r, l, t, mo, y]
                    * m.TechnologyFromStorage[r, t, s, mo]
                    * m.Conversionls[l, ls]
                    * m.Conversionld[l, ld]
                    * m.Conversionlh[l, lh]
                    for mo in m.MODE_OF_OPERATION
                    for l in m.TIMESLICE
                    for t in m.TECHNOLOGY
                )
                == m.RateOfStorageDischarge[r, s, ls, ld, lh, y]
            )
        return Constraint.Skip
    model.RateOfStorageDischarge_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR, model.TECHNOLOGY, model.MODE_OF_OPERATION,
        rule=RateOfStorageDischarge_rule,
    )

    def NetChargeWithinYear_rule(m, r, s, ls, ld, lh, y):
        return (
            sum(
                (m.RateOfStorageCharge[r, s, ls, ld, lh, y]
                 - m.RateOfStorageDischarge[r, s, ls, ld, lh, y])
                * m.YearSplit[l, y]
                * m.Conversionls[l, ls]
                * m.Conversionld[l, ld]
                * m.Conversionlh[l, lh]
                for l in m.TIMESLICE
            )
            == m.NetChargeWithinYear[r, s, ls, ld, lh, y]
        )
    model.NetChargeWithinYear_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=NetChargeWithinYear_rule,
    )

    def NetChargeWithinDay_rule(m, r, s, ls, ld, lh, y):
        return (
            (m.RateOfStorageCharge[r, s, ls, ld, lh, y]
             - m.RateOfStorageDischarge[r, s, ls, ld, lh, y])
            * m.DaySplit[lh, y]
            == m.NetChargeWithinDay[r, s, ls, ld, lh, y]
        )
    model.NetChargeWithinDay_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=NetChargeWithinDay_rule,
    )

    # --- Storage level tracking ---

    def StorageLevelYearStart_rule(m, r, s, y):
        if y == min(m.YEAR):
            return m.StorageLevelStart[r, s] == m.StorageLevelYearStart[r, s, y]
        return (
            m.StorageLevelYearStart[r, s, y - 1]
            + sum(
                m.NetChargeWithinYear[r, s, ls, ld, lh, y - 1]
                for ls in m.SEASON for ld in m.DAYTYPE for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelYearStart[r, s, y]
        )
    model.StorageLevelYearStart_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=StorageLevelYearStart_rule,
    )

    def StorageLevelYearFinish_rule(m, r, s, y):
        if y < max(m.YEAR):
            return m.StorageLevelYearStart[r, s, y + 1] == m.StorageLevelYearFinish[r, s, y]
        return (
            m.StorageLevelYearStart[r, s, y]
            + sum(
                m.NetChargeWithinYear[r, s, ls, ld, lh, y - 1]
                for ls in m.SEASON for ld in m.DAYTYPE for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelYearFinish[r, s, y]
        )
    model.StorageLevelYearFinish_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=StorageLevelYearFinish_rule,
    )

    def StorageLevelSeasonStart_rule(m, r, s, ls, y):
        if ls == min(m.SEASON):
            return m.StorageLevelYearStart[r, s, y] == m.StorageLevelSeasonStart[r, s, ls, y]
        return (
            m.StorageLevelSeasonStart[r, s, ls - 1, y]
            + sum(
                m.NetChargeWithinYear[r, s, ls - 1, ld, lh, y]
                for ld in m.DAYTYPE for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelSeasonStart[r, s, ls, y]
        )
    model.StorageLevelSeasonStart_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.YEAR,
        rule=StorageLevelSeasonStart_rule,
    )

    def StorageLevelDayTypeStart_rule(m, r, s, ls, ld, y):
        if ld == min(m.DAYTYPE):
            return (
                m.StorageLevelSeasonStart[r, s, ls, y]
                == m.StorageLevelDayTypeStart[r, s, ls, ld, y]
            )
        return (
            m.StorageLevelDayTypeStart[r, s, ls, ld - 1, y]
            + sum(
                m.NetChargeWithinDay[r, s, ls, ld - 1, lh, y]
                for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelDayTypeStart[r, s, ls, ld, y]
        )
    model.StorageLevelDayTypeStart_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
        rule=StorageLevelDayTypeStart_rule,
    )

    def StorageLevelDayTypeFinish_rule(m, r, s, ls, ld, y):
        if ld == max(m.DAYTYPE):
            if ls == max(m.SEASON):
                return (
                    m.StorageLevelYearFinish[r, s, y]
                    == m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
                )
            return (
                m.StorageLevelSeasonStart[r, s, ls + 1, y]
                == m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
            )
        return (
            m.StorageLevelDayTypeFinish[r, s, ls, ld + 1, y]
            - sum(
                m.NetChargeWithinDay[r, s, ls, ld + 1, lh, y]
                for lh in m.DAILYTIMEBRACKET
            )
            * m.DaysInDayType[ls, ld + 1, y]
            == m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
        )
    model.StorageLevelDayTypeFinish_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
        rule=StorageLevelDayTypeFinish_rule,
    )

    # --- Storage bounds ---

    def LowerLimit_1TimeBracket1InstanceOfDayType1week_rule(m, r, s, ls, ld, lh, y):
        return (
            0
            <= (
                m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                + sum(
                    m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                )
            )
            - m.StorageLowerLimit[r, s, y]
        )
    model.LowerLimit_1TimeBracket1InstanceOfDayType1week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_1TimeBracket1InstanceOfDayType1week_rule,
    )

    def LowerLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                0
                <= (
                    m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                    - sum(
                        m.NetChargeWithinDay[r, s, ls, ld - 1, lhlh, y]
                        for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
                    )
                )
                - m.StorageLowerLimit[r, s, y]
            )
        return Constraint.Skip
    model.LowerLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule,
    )

    def LowerLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule(m, r, s, ls, ld, lh, y):
        return (
            0
            <= (
                m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
                - sum(
                    m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
                )
            )
            - m.StorageLowerLimit[r, s, y]
        )
    model.LowerLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule,
    )

    def LowerLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                0
                <= (
                    m.StorageLevelDayTypeFinish[r, s, ls, ld - 1, y]
                    + sum(
                        m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                        for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                    )
                )
                - m.StorageLowerLimit[r, s, y]
            )
        return Constraint.Skip
    model.LowerLimit_1TimeBracket1InstanceOfDayTypeLastweek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule,
    )

    def UpperLimit_1TimeBracket1InstanceOfDayType1week_rule(m, r, s, ls, ld, lh, y):
        return (
            (
                m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                + sum(
                    m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                )
            )
            - m.StorageUpperLimit[r, s, y]
            <= 0
        )
    model.UpperLimit_1TimeBracket1InstanceOfDayType1week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_1TimeBracket1InstanceOfDayType1week_rule,
    )

    def UpperLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                - sum(
                    m.NetChargeWithinDay[r, s, ls, ld - 1, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
                )
            ) - m.StorageUpperLimit[r, s, y] <= 0
        return Constraint.Skip
    model.UpperLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule,
    )

    def UpperLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule(m, r, s, ls, ld, lh, y):
        return (
            m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
            - sum(
                m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
            )
        ) - m.StorageUpperLimit[r, s, y] <= 0
    model.UpperLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule,
    )

    def UpperLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                0
                >= (
                    m.StorageLevelDayTypeFinish[r, s, ls, ld - 1, y]
                    + sum(
                        m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                        for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                    )
                )
                - m.StorageUpperLimit[r, s, y]
            )
        return Constraint.Skip
    model.UpperLimit_1TimeBracket1InstanceOfDayTypeLastweek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule,
    )

    # --- Charge/discharge rate limits ---

    def MaxChargeConstraint_rule(m, r, s, ls, ld, lh, y):
        return m.RateOfStorageCharge[r, s, ls, ld, lh, y] <= m.StorageMaxChargeRate[r, s]
    model.MaxChargeConstraint_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=MaxChargeConstraint_rule,
    )

    def MaxDischargeConstraint_rule(m, r, s, ls, ld, lh, y):
        return m.RateOfStorageDischarge[r, s, ls, ld, lh, y] <= m.StorageMaxDischargeRate[r, s]
    model.MaxDischargeConstraint_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=MaxDischargeConstraint_rule,
    )

    # --- Storage investments ---

    def StorageUpperLimit_rule(m, r, s, y):
        return (
            m.AccumulatedNewStorageCapacity[r, s, y]
            + m.ResidualStorageCapacity[r, s, y]
            == m.StorageUpperLimit[r, s, y]
        )
    model.StorageUpperLimit_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR, rule=StorageUpperLimit_rule,
    )

    def StorageLowerLimit_rule(m, r, s, y):
        return (
            m.MinStorageCharge[r, s, y] * m.StorageUpperLimit[r, s, y]
            == m.StorageLowerLimit[r, s, y]
        )
    model.StorageLowerLimit_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR, rule=StorageLowerLimit_rule,
    )

    def TotalNewStorage_rule(m, r, s, y):
        return (
            sum(
                m.NewStorageCapacity[r, s, yy]
                for yy in m.YEAR
                if ((y - yy < m.OperationalLifeStorage[r, s]) and (y - yy >= 0))
            )
            == m.AccumulatedNewStorageCapacity[r, s, y]
        )
    model.TotalNewStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR, rule=TotalNewStorage_rule,
    )

    def UndiscountedCapitalInvestmentStorage_rule(m, r, s, y):
        return (
            m.CapitalCostStorage[r, s, y] * m.NewStorageCapacity[r, s, y]
            == m.CapitalInvestmentStorage[r, s, y]
        )
    model.UndiscountedCapitalInvestmentStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=UndiscountedCapitalInvestmentStorage_rule,
    )

    def DiscountingCapitalInvestmentStorage_rule(m, r, s, y):
        return (
            m.CapitalInvestmentStorage[r, s, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR)))
            == m.DiscountedCapitalInvestmentStorage[r, s, y]
        )
    model.DiscountingCapitalInvestmentStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=DiscountingCapitalInvestmentStorage_rule,
    )

    def SalvageValueStorageAtEndOfPeriod_rule(m, r, s, y):
        if (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLifeStorage[r, s] - 1) > max(m.YEAR))
            and m.DiscountRate[r] > 0
        ):
            return m.SalvageValueStorage[r, s, y] == m.CapitalInvestmentStorage[r, s, y] * (
                1 - (
                    ((1 + m.DiscountRate[r]) ** (max(m.YEAR) - y + 1) - 1)
                    / ((1 + m.DiscountRate[r]) ** m.OperationalLifeStorage[r, s] - 1)
                )
            )
        elif (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLifeStorage[r, s] - 1) > max(m.YEAR))
            and m.DiscountRate[r] == 0
        ) or (
            m.DepreciationMethod[r] == 2
            and (y + m.OperationalLifeStorage[r, s] - 1) > max(m.YEAR)
        ):
            return m.SalvageValueStorage[r, s, y] == m.CapitalInvestmentStorage[r, s, y] * (
                1 - (max(m.YEAR) - y + 1) / m.OperationalLifeStorage[r, s]
            )
        return m.SalvageValueStorage[r, s, y] == 0
    model.SalvageValueStorageAtEndOfPeriod_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=SalvageValueStorageAtEndOfPeriod_rule,
    )

    def SalvageValueStorageDiscountedToStartYear_rule(m, r, s, y):
        return (
            m.SalvageValueStorage[r, s, y]
            / ((1 + m.DiscountRate[r]) ** (max(m.YEAR) - min(m.YEAR) + 1))
            == m.DiscountedSalvageValueStorage[r, s, y]
        )
    model.SalvageValueDiscountedToStartYear_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=SalvageValueStorageDiscountedToStartYear_rule,
    )

    def TotalDiscountedCostByStorage_rule(m, r, s, y):
        return (
            m.DiscountedCapitalInvestmentStorage[r, s, y]
            - m.DiscountedSalvageValueStorage[r, s, y]
            == m.TotalDiscountedStorageCost[r, s, y]
        )
    model.TotalDiscountedCostByStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=TotalDiscountedCostByStorage_rule,
    )

In [ ]:
from pyomo.environ import Set, Param, Var, Constraint, Objective, DataPortal

def _csv_has_data(path):
    return path.is_file() and path.stat().st_size > 7

has_storage = _csv_has_data(csv_dir / "STORAGE.csv")
has_udc = _csv_has_data(csv_dir / "UDC.csv")
print(f"has_storage = {has_storage}")
print(f"has_udc     = {has_udc}")

model = create_abstract_model(has_storage=has_storage, has_udc=has_udc)
print(f"AbstractModel: Sets={sum(1 for _ in model.component_objects(Set, active=True))}, "
      f"Params={sum(1 for _ in model.component_objects(Param, active=True))}, "
      f"Vars={sum(1 for _ in model.component_objects(Var, active=True))}, "
      f"Constraints={sum(1 for _ in model.component_objects(Constraint, active=True))}")

## 6. Construir la instancia

In [ ]:
def _load_set(data, p, fname, sn):
    fp = os.path.join(p, fname)
    if not os.path.exists(fp): return
    with open(fp, encoding="utf-8") as f:
        f.readline(); first = f.readline().strip()
    if not first: return
    data.load(filename=fp, set=sn)

def _load_param(data, p, fname, pn, idx):
    fp = os.path.join(p, fname)
    if not os.path.exists(fp): return
    with open(fp, encoding="utf-8") as f:
        f.readline(); first = f.readline().strip()
    if not first: return
    data.load(filename=fp, param=pn, index=idx)


def build_instance_inline(model, csv_dir, *, has_storage=False, has_udc=False):
    data = DataPortal(); p = str(csv_dir)
    for fname, sn in [("EMISSION.csv","EMISSION"),("FUEL.csv","FUEL"),("TIMESLICE.csv","TIMESLICE"),
                      ("MODE_OF_OPERATION.csv","MODE_OF_OPERATION"),("TECHNOLOGY.csv","TECHNOLOGY"),
                      ("YEAR.csv","YEAR"),("REGION.csv","REGION")]:
        _load_set(data, p, fname, sn)
    if has_storage:
        for fname, sn in [("STORAGE.csv","STORAGE"),("SEASON.csv","SEASON"),
                          ("DAYTYPE.csv","DAYTYPE"),("DAILYTIMEBRACKET.csv","DAILYTIMEBRACKET")]:
            _load_set(data, p, fname, sn)
    PARAMS = [
        ("YearSplit.csv","YearSplit",["TIMESLICE","YEAR"]),
        ("DiscountRate.csv","DiscountRate",["REGION"]),
        ("DepreciationMethod.csv","DepreciationMethod",["REGION"]),
        ("CapacityToActivityUnit.csv","CapacityToActivityUnit",["REGION","TECHNOLOGY"]),
        ("CapacityOfOneTechnologyUnit.csv","CapacityOfOneTechnologyUnit",["REGION","TECHNOLOGY","YEAR"]),
        ("OperationalLife.csv","OperationalLife",["REGION","TECHNOLOGY"]),
        ("TotalAnnualMaxCapacityInvestment.csv","TotalAnnualMaxCapacityInvestment",["REGION","TECHNOLOGY","YEAR"]),
        ("TotalAnnualMinCapacityInvestment.csv","TotalAnnualMinCapacityInvestment",["REGION","TECHNOLOGY","YEAR"]),
        ("TotalTechnologyAnnualActivityLowerLimit.csv","TotalTechnologyAnnualActivityLowerLimit",["REGION","TECHNOLOGY","YEAR"]),
        ("TotalTechnologyAnnualActivityUpperLimit.csv","TotalTechnologyAnnualActivityUpperLimit",["REGION","TECHNOLOGY","YEAR"]),
        ("TotalTechnologyModelPeriodActivityLowerLimit.csv","TotalTechnologyModelPeriodActivityLowerLimit",["REGION","TECHNOLOGY"]),
        ("TotalTechnologyModelPeriodActivityUpperLimit.csv","TotalTechnologyModelPeriodActivityUpperLimit",["REGION","TECHNOLOGY"]),
        ("CapacityFactor.csv","CapacityFactor",["REGION","TECHNOLOGY","TIMESLICE","YEAR"]),
        ("AvailabilityFactor.csv","AvailabilityFactor",["REGION","TECHNOLOGY","YEAR"]),
        ("ResidualCapacity.csv","ResidualCapacity",["REGION","TECHNOLOGY","YEAR"]),
        ("CapitalCost.csv","CapitalCost",["REGION","TECHNOLOGY","YEAR"]),
        ("FixedCost.csv","FixedCost",["REGION","TECHNOLOGY","YEAR"]),
        ("VariableCost.csv","VariableCost",["REGION","TECHNOLOGY","MODE_OF_OPERATION","YEAR"]),
        ("EmissionActivityRatio.csv","EmissionActivityRatio",["REGION","TECHNOLOGY","EMISSION","MODE_OF_OPERATION","YEAR"]),
        ("EmissionsPenalty.csv","EmissionsPenalty",["REGION","EMISSION","YEAR"]),
        ("ModelPeriodEmissionLimit.csv","ModelPeriodEmissionLimit",["REGION","EMISSION"]),
        ("ModelPeriodExogenousEmission.csv","ModelPeriodExogenousEmission",["REGION","EMISSION"]),
        ("AnnualExogenousEmission.csv","AnnualExogenousEmission",["REGION","EMISSION","YEAR"]),
        ("AnnualEmissionLimit.csv","AnnualEmissionLimit",["REGION","EMISSION","YEAR"]),
        ("InputActivityRatio.csv","InputActivityRatio",["REGION","TECHNOLOGY","FUEL","MODE_OF_OPERATION","YEAR"]),
        ("OutputActivityRatio.csv","OutputActivityRatio",["REGION","TECHNOLOGY","FUEL","MODE_OF_OPERATION","YEAR"]),
        ("ReserveMarginTagFuel.csv","ReserveMarginTagFuel",["REGION","FUEL","YEAR"]),
        ("RETagTechnology.csv","RETagTechnology",["REGION","TECHNOLOGY","YEAR"]),
        ("RETagFuel.csv","RETagFuel",["REGION","FUEL","YEAR"]),
        ("REMinProductionTarget.csv","REMinProductionTarget",["REGION","YEAR"]),
        ("ReserveMarginTagTechnology.csv","ReserveMarginTagTechnology",["REGION","TECHNOLOGY","YEAR"]),
        ("ReserveMargin.csv","ReserveMargin",["REGION","YEAR"]),
        ("AccumulatedAnnualDemand.csv","AccumulatedAnnualDemand",["REGION","FUEL","YEAR"]),
        ("SpecifiedAnnualDemand.csv","SpecifiedAnnualDemand",["REGION","FUEL","YEAR"]),
        ("SpecifiedDemandProfile.csv","SpecifiedDemandProfile",["REGION","FUEL","TIMESLICE","YEAR"]),
        ("TotalAnnualMaxCapacity.csv","TotalAnnualMaxCapacity",["REGION","TECHNOLOGY","YEAR"]),
        ("TotalAnnualMinCapacity.csv","TotalAnnualMinCapacity",["REGION","TECHNOLOGY","YEAR"]),
    ]
    for f, n, i in PARAMS: _load_param(data, p, f, n, i)
    return model.create_instance(data)


t0 = time.perf_counter()
instance = build_instance_inline(model, csv_dir, has_storage=has_storage, has_udc=has_udc)
t_build = time.perf_counter() - t0
n_v = sum(len(v) for v in instance.component_objects(Var, active=True))
n_c = sum(len(c) for c in instance.component_objects(Constraint, active=True))
print(f"Instancia construida en {t_build:.2f} s ({n_v} vars, {n_c} cons)")
print(f"  instance.YEAR: {sorted(int(y) for y in instance.YEAR)[:3]}...{sorted(int(y) for y in instance.YEAR)[-3:]}")

## 7. Resolver (Gurobi → HiGHS → GLPK)

In [ ]:
import time, gc
import pyomo.environ as pyo
from pyomo.opt import SolverFactory


def _gurobi_lightweight_available():
    try: import gurobipy
    except Exception: return False
    return True


def _is_available(name):
    if name == "gurobi": return _gurobi_lightweight_available()
    try: return bool(SolverFactory(name).available(exception_flag=False))
    except Exception: return False


def _dispose_gurobi():
    try:
        import gurobipy as gp
        if hasattr(gp, "disposeDefaultEnv"): gp.disposeDefaultEnv()
    except Exception: pass
    gc.collect()


SOLVERS_PRIORITY = ["gurobi", "appsi_highs", "glpk"]
obj_val = None; results = None; solver_name_used = None; t_solve = None

for sname in SOLVERS_PRIORITY:
    if not _is_available(sname):
        print(f"  {sname}: no disponible, salteando")
        continue
    print(f"\n=== Intentando {sname} ===")
    try:
        s = SolverFactory(sname)
        t0 = time.perf_counter()
        res = s.solve(instance, tee=True, load_solutions=False)
        t_solve = time.perf_counter() - t0
        raw = str(res.solver.termination_condition)
        print(f"\n  status={res.solver.status} termination={raw} elapsed={t_solve:.2f}s")
        if "optimal" in raw.lower():
            instance.solutions.load_from(res)
            for o in instance.component_objects(pyo.Objective, active=True):
                obj_val = float(pyo.value(o)); break
            results = res; solver_name_used = sname
            print(f"  objective = {obj_val:.6e}")
            if sname == "gurobi": _dispose_gurobi()
            break
        else:
            results = res; solver_name_used = sname
            if sname == "gurobi": _dispose_gurobi()
            break
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {str(e)[:200]}")
        if sname == "gurobi": _dispose_gurobi()

if obj_val is None:
    print(f"\n!! No se obtuvo solucion optima.")
else:
    print(f"\nOK con {solver_name_used} en {t_solve:.2f}s, obj={obj_val:.6e}")

## 8. Diagnóstico de infactibilidad

In [ ]:
from pyomo.core import Constraint, Var, value
from collections import Counter

def run_infeasibility_diagnostics(instance, *, top_n=10, tol=1e-6):
    print("=" * 70); print("DIAGNOSTICO DE INFACTIBILIDAD"); print("=" * 70)
    cv = []
    for c in instance.component_data_objects(Constraint, active=True):
        b = value(c.body, exception=False)
        if b is None: continue
        lb = value(c.lower, exception=False) if c.has_lb() else None
        ub = value(c.upper, exception=False) if c.has_ub() else None
        vio = 0.0; side = ""
        if lb is not None and b < lb - tol: vio = lb - b; side = "LB"
        elif ub is not None and b > ub + tol: vio = b - ub; side = "UB"
        if vio > tol: cv.append((c.name, b, lb, ub, side, vio))
    cv.sort(key=lambda x: -x[5])

    bc = []
    for v in instance.component_data_objects(Var, active=True):
        lb = value(v.lb, exception=False) if v.has_lb() else None
        ub = value(v.ub, exception=False) if v.has_ub() else None
        if lb is not None and ub is not None and lb > ub + tol:
            bc.append((v.name, lb, ub, lb - ub))
    bc.sort(key=lambda x: -x[3])

    print(f"\nRestricciones violadas: {len(cv)}")
    if cv:
        for n, b, lb, ub, s, v in cv[:top_n]:
            print(f"  {n[:60]:<60s} body={b:.3e} viol={v:.3e} ({s})")
        fam = Counter(name.split("[")[0] for name, *_ in cv)
        print(f"\n  Top familias:")
        for f, n in fam.most_common(8): print(f"    {f:<45s} {n}")

    print(f"\nVariables LB > UB: {len(bc)}")
    return {"constraint_violations": cv, "var_bound_conflicts": bc}


if results is not None and obj_val is None:
    diag = run_infeasibility_diagnostics(instance)
elif obj_val is not None:
    print("Solucion optima encontrada; sin diagnostico de infactibilidad.")

## 9. Resultados como DataFrames (estilo data explorer)

In [ ]:
import pandas as pd
from pyomo.core import Var

def instance_to_long_df(instance, *, var_names=None, threshold=1e-9):
    rows = []
    targets = list(instance.component_objects(Var, active=True))
    if var_names: targets = [v for v in targets if v.name in var_names]
    for v in targets:
        for k in v:
            val = v[k].value
            if val is None or abs(val) < threshold: continue
            if not isinstance(k, tuple): k = (k,)
            row = {"variable": v.name, "value": float(val)}
            for i, x in enumerate(k): row[f"k{i}"] = x
            rows.append(row)
    return pd.DataFrame(rows)


if obj_val is not None:
    df_long = instance_to_long_df(instance)
    print(f"Total filas con valor != 0: {len(df_long)}")
    print(f"\nVariables presentes (top 10 por # filas):")
    print(df_long["variable"].value_counts().head(10).to_string())

    # Top 20 valores absolutos
    print(f"\nTop 20 valores absolutos:")
    print(df_long.reindex(df_long["value"].abs().sort_values(ascending=False).index)
          .head(20).to_string(index=False))
else:
    print("No hay solucion para inspeccionar.")